# Loading Data And QC

**REQUIRED DAY 2**

## From count matrix to AnnData

Everything from here on works with **AnnData** — the data structure `scanpy` is built on: a cell-by-gene matrix (`.X`) plus per-cell metadata (`.obs`) and per-gene metadata (`.var`) that travel together as one object.

In [ ]:
import scanpy as sc

adata = sc.read_h5ad("/tscc/nfs/home/juf009/day2_shared_data/counts/checkpoint.h5ad")
adata

In [ ]:
adata.obs_names[:5]   # cell barcodes

In [ ]:
adata.var_names[:5]   # gene symbols

Confirm `adata.n_obs` and `adata.n_vars` are in the range you expect *before* doing anything else — this is exactly what [templates/diagnostic_scripts/verify_counts_matrix.py](../templates/diagnostic_scripts/verify_counts_matrix.py) automates.

## Quality control metrics

Per-cell QC in scanpy centers on three numbers:

- **`n_genes_by_counts`** — how many distinct genes were detected in this cell. Very low: likely an empty droplet or dying cell. Very high: possibly a doublet (see below).
- **`total_counts`** — total UMIs per cell. Same logic as above.
- **`pct_counts_mt`** — percent of counts from mitochondrial genes. High mitochondrial fraction is a classic signature of a dying or ruptured cell (cytoplasmic RNA leaks out, mitochondrial RNA is relatively retained).

In [ ]:
adata.var["mt"] = adata.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True)
adata.obs[["n_genes_by_counts", "total_counts", "pct_counts_mt"]].describe()

In [ ]:
sc.pl.violin(adata, ["n_genes_by_counts", "total_counts", "pct_counts_mt"], jitter=0.4)

## Doublet flagging is a QC step, not an afterthought

A "doublet" is two cells captured in one droplet and sequenced as if they were one cell — it will look like a real cell with QC metrics that pass every threshold above, but its transcriptome is a mixture of two cell types. Flag it explicitly rather than hoping clustering will sort it out later:

In [ ]:
sc.pp.scrublet(adata)
adata.obs["predicted_doublet"].value_counts()

Doublets don't announce themselves in the QC metrics above — a doublet can pass every threshold and still be two cells' worth of transcriptome mixed together. Checking for it explicitly, rather than assuming clustering will sort it out later, is the point of this step.

## Practice

Look at the histograms above yourself and write your own filtering thresholds for `n_genes_by_counts`, `total_counts`, and `pct_counts_mt`, with one sentence of reasoning for each based on what you actually see in *this* dataset's distributions — not a number copied from a tutorial. Apply them in the cell below.

In [ ]:
# Apply your QC filtering thresholds here, once you've written down your reasoning above.


## Save your checkpoint

Each notebook in this sequence opens with its own fresh kernel, so `adata` doesn't carry over by itself -- this saves it to disk, and 06 loads it back in. Same reasoning as every checkpoint file elsewhere in this bootcamp.

In [ ]:
import os

os.makedirs("results", exist_ok=True)
adata.write_h5ad("results/checkpoint_05_qc.h5ad")
print("Saved to results/checkpoint_05_qc.h5ad")


## Further reading

- [Single-cell best practices — Quality Control](https://www.sc-best-practices.org/preprocessing_visualization/quality_control.html)
- [scanpy: Preprocessing and clustering tutorial](https://scanpy.readthedocs.io/en/stable/tutorials/basics/clustering.html)